# Milestone 2: Data Quality dan Profiling

Lanjutkan dari hasil `01_data_profiling.ipynb`. Notebook ini menerapkan
profiling DuckDB `SUMMARIZE` dan transformasi Polars untuk menangani missing
values, outlier, duplikat, serta validasi data dasar.

Dataset yang dipakai sama dengan Milestone 1: berita politik Indonesia,
640.000 baris dan 10 kolom. Hasil pembersihan disimpan ke
`data/processed/berita_politik_bersih.parquet`.

In [1]:
from pathlib import Path

import duckdb
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'berita_politik_640rb.parquet'
OUTPUT_PATH = PROJECT_ROOT / 'data' / 'processed' / 'berita_politik_bersih.parquet'

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'Dataset tidak ditemukan: {DATA_PATH}. '
        'Jalankan python src/download_dataset.py lebih dulu.'
    )

df = pl.read_parquet(DATA_PATH)
print(f'Ukuran awal: {df.shape[0]:,} baris x {df.shape[1]} kolom')
df.head()

Ukuran awal: 640,000 baris x 10 kolom


id,news_title,news_source,news_author,news_hostname,news_date,news_text,news_tags,news_kabkot,news_intent
str,str,str,str,str,str,str,str,str,str
"""1924973""","""Dana Kegiatan Isbat Nikah Baka…","""https://radarutara.disway.id/m…","""Wahyudi""","""disway.id""","""2026-04-27 01:30:00+00:00""","""Dana Kegiatan Isbat Nikah Baka…","""['apbd perubahan 2026', 'diaju…","""Kabupaten Mukomuko""","""{'intent': 'lapor_informasi', …"
"""1925951""","""Satu Orang Meninggal dari Insi…","""https://www.ayokaltim.com/satu…","""Iswanto""","""ayokaltim.com""","""2026-04-27 03:19:46+00:00""","""Satu Orang Meninggal dari Insi…","""['Peristiwa di Samarinda;Tabra…","""Kota Samarinda""","""{'intent': 'lapor_informasi', …"
"""1925952""","""DPRD Bongkar Ketidaksesuaian L…","""https://infosatu.co/dprd-bongk…","""Emmy Haryanti""","""infosatu.co""","""2026-04-27 03:59:29+00:00""","""Samarinda, infosatu.co – Klaim…","""['Abdul rohim;DPRD Samarinda;L…","""Kota Samarinda""","""{'intent': 'lapor_informasi', …"
"""1925953""","""DPRD Dorong Penanganan Banjir …","""https://infosatu.co/dprd-doron…","""Emmy Haryanti""","""infosatu.co""","""2026-04-27 01:43:24+00:00""","""Samarinda, infosatu.co – Dewan…","""['Banjir;Deni Hakim Anwar;DPRD…","""Kota Samarinda""","""{'intent': 'lapor_informasi', …"
"""1924974""","""Kepung Pasar Karang Nabire, Ma…","""https://papuatengah.tribunnews…","""Calvin Eluis Erari""","""tribunnews.com""","""2026-04-27 02:14:39+00:00""","""Demo di Nabire Kepung Pasar Ka…","""[]""","""Kabupaten Nabire""","""{'intent': 'lapor_informasi', …"


## 1. Profiling dengan DuckDB `SUMMARIZE`

Dapatkan ringkasan statistik seluruh kolom (tipe data, null%, min/max, distinct count) dalam satu perintah.

In [2]:
connection = duckdb.connect()
connection.register('dataset', df)

profil = connection.execute('SUMMARIZE SELECT * FROM dataset').pl()
profil

column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
str,str,str,str,i64,i32,i32,i32,i32,i32,i64,"decimal[9,2]"
"""id""","""VARCHAR""","""1""","""99996""",642240,null,null,null,null,null,640000,0.00
"""news_title""","""VARCHAR""",""" - MediaPATRIOT.CO.ID""","""🪨 Produksi Batu Bara 2025 dan …",617656,null,null,null,null,null,640000,0.00
"""news_source""","""VARCHAR""","""http://aceh.navigasi.co.id/amp…","""https://zuper.fajar.co.id/2026…",681664,null,null,null,null,null,640000,0.00
"""news_author""","""VARCHAR""","""( )""","""𝗗𝟯𝗹 𝟬𝟮""",35718,null,null,null,null,null,640000,4.99
"""news_hostname""","""VARCHAR""","""1kabar.com""","""zuper.fajar.co.id""",5065,null,null,null,null,null,640000,0.00
"""news_date""","""VARCHAR""","""0001-11-30 00:00:00+00:00""","""2265-10-25 17:00:00+00:00""",266028,null,null,null,null,null,640000,2.60
"""news_text""","""VARCHAR""","""! Banyuwangi, Jatim Hari Ini -…","""🔥 KRITIK LGI MAKIN PEDAS: Real…",664842,null,null,null,null,null,640000,0.00
"""news_tags""","""VARCHAR""","""[""#KapolresPangandaran;#poldaj…","""[]""",398837,null,null,null,null,null,640000,0.00
"""news_kabkot""","""VARCHAR""","""Aceh Barat""","""Yalimo""",1044,null,null,null,null,null,640000,0.00


## 2. Deteksi Missing Values

Lihat jumlah dan persentase null per kolom sebelum memutuskan strategi penanganan.

In [3]:
jumlah_baris = len(df)
null_summary = (
    df.null_count()
    .transpose(include_header=True, header_name='kolom', column_names=['jumlah_null'])
    .with_columns(
        (pl.col('jumlah_null') / jumlah_baris * 100).round(2).alias('persen_null')
    )
    .filter(pl.col('jumlah_null') > 0)
    .sort('jumlah_null', descending=True)
)
null_summary

kolom,jumlah_null,persen_null
str,u32,f64
"""news_author""",31931,4.99
"""news_date""",16624,2.6


## 3. Penanganan Missing Values

Hanya dua kolom yang punya nilai kosong, dan keduanya diperlakukan berbeda
karena jenis datanya berbeda.

`news_author` bertipe teks. Nilainya diisi `Tanpa Penulis`, bukan dibuang,
karena penulis yang tidak tercatat tetap berita yang sah dan kolom ini tidak
dipakai sebagai kunci analisis.

`news_date` bertipe waktu. Kolom ini tidak diisi nilai tebakan. Tanggal terbit
yang dikarang akan merusak analisis temporal di Milestone 3, karena barisnya
akan tampak berasal dari periode tertentu padahal tidak diketahui. Yang
dilakukan: nilainya diubah ke datetime, lalu ditandai lewat kolom
`tanggal_valid`. Baris dengan tanggal kosong atau di luar rentang 2015-2026
dibiarkan null pada `tanggal_terbit` dan ditandai `False`.

In [4]:
df_missing_handled = df.with_columns([
    # Kategorik: isi dengan penanda, bukan dibuang
    pl.col('news_author').fill_null('Tanpa Penulis'),

    # Waktu: ubah ke datetime, jangan diisi nilai tebakan
    pl.col('news_date')
    .str.to_datetime(strict=False, time_zone='UTC')
    .alias('tanggal_terbit'),
])

# Tanggal di luar rentang wajar dijadikan null, lalu ditandai
df_missing_handled = df_missing_handled.with_columns(
    pl.when(pl.col('tanggal_terbit').dt.year().is_between(2015, 2026))
    .then(pl.col('tanggal_terbit'))
    .otherwise(None)
    .alias('tanggal_terbit')
).with_columns(
    pl.col('tanggal_terbit').is_not_null().alias('tanggal_valid')
)

print(f'Null sebelum: {df.null_count().sum_horizontal().sum():,}')
print(f'Null sesudah: {df_missing_handled.null_count().sum_horizontal().sum():,}')
print()
print('Rincian:')
print(f"  news_author  : {df['news_author'].null_count():,} -> {df_missing_handled['news_author'].null_count():,}")
print(f"  news_date    : {df['news_date'].null_count():,} -> {df_missing_handled['news_date'].null_count():,}")
print(f"  tanggal_terbit null : {df_missing_handled['tanggal_terbit'].null_count():,}")
print(f"  tanggal_valid False : {(~df_missing_handled['tanggal_valid']).sum():,}")

Null sebelum: 48,555
Null sesudah: 33,364

Rincian:
  news_author  : 31,931 -> 0
  news_date    : 16,624 -> 16,624
  tanggal_terbit null : 16,740
  tanggal_valid False : 16,740


## 4. Deteksi dan Penanganan Outlier

Gunakan metode IQR sebagai titik awal, lalu kombinasikan dengan batas bisnis
(domain knowledge) yang relevan dengan dataset Anda.

Kolom numerik yang diperiksa adalah panjang isi berita. Panjang teks dipakai
sebagai penanda kualitas: artikel yang terlalu pendek biasanya hasil ekstraksi
gagal, sedangkan yang terlalu panjang biasanya gabungan beberapa halaman.

In [5]:
df_teks = df_missing_handled.with_columns([
    pl.col('news_text').str.len_chars().alias('panjang_teks'),
    pl.col('news_title').str.len_chars().alias('panjang_judul'),
])

KOLOM_NUMERIK = 'panjang_teks'

q1 = df_teks[KOLOM_NUMERIK].quantile(0.25)
q3 = df_teks[KOLOM_NUMERIK].quantile(0.75)
iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr

print(f'Q1={q1:,.0f}, Q3={q3:,.0f}, IQR={iqr:,.0f}')
print(f'Batas bawah IQR: {batas_bawah:,.0f}, Batas atas IQR: {batas_atas:,.0f}')

n_bawah_iqr = (df_teks[KOLOM_NUMERIK] < batas_bawah).sum()
n_atas_iqr = (df_teks[KOLOM_NUMERIK] > batas_atas).sum()
print(f'Terdeteksi IQR: {n_bawah_iqr:,} di bawah, {n_atas_iqr:,} di atas')

# Batas bisnis: artikel di bawah 100 karakter dianggap ekstraksi gagal,
# di atas 50.000 karakter dianggap gabungan beberapa halaman.
batas_bawah_bisnis = 100
batas_atas_bisnis = 50_000

df_outlier_handled = df_teks.with_columns([
    pl.when(
        (pl.col(KOLOM_NUMERIK) < batas_bawah_bisnis)
        | (pl.col(KOLOM_NUMERIK) > batas_atas_bisnis)
    )
    .then(pl.lit(True))
    .otherwise(pl.lit(False))
    .alias('is_outlier_teks'),

    # Windsorization pada kolom turunan, isi berita aslinya tidak diubah
    pl.col(KOLOM_NUMERIK)
    .clip(lower_bound=batas_bawah_bisnis, upper_bound=batas_atas_bisnis)
    .alias('panjang_teks_bersih'),
])

n_outlier = df_outlier_handled['is_outlier_teks'].sum()
print(f'Outlier batas bisnis: {n_outlier:,} baris ({n_outlier / len(df_outlier_handled) * 100:.4f}%)')
print()
print(df_outlier_handled.select(['panjang_teks', 'panjang_teks_bersih']).describe())

Q1=1,280, Q3=2,533, IQR=1,253
Batas bawah IQR: -600, Batas atas IQR: 4,412
Terdeteksi IQR: 0 di bawah, 18,816 di atas
Outlier batas bisnis: 4 baris (0.0006%)

shape: (9, 3)
┌────────────┬──────────────┬─────────────────────┐
│ statistic  ┆ panjang_teks ┆ panjang_teks_bersih │
│ ---        ┆ ---          ┆ ---                 │
│ str        ┆ f64          ┆ f64                 │
╞════════════╪══════════════╪═════════════════════╡
│ count      ┆ 640000.0     ┆ 640000.0            │
│ null_count ┆ 0.0          ┆ 0.0                 │
│ mean       ┆ 2006.946166  ┆ 2005.163373         │
│ std        ┆ 1508.426721  ┆ 1219.691016         │
│ min        ┆ 176.0        ┆ 176.0               │
│ 25%        ┆ 1280.0       ┆ 1280.0              │
│ 50%        ┆ 1897.0       ┆ 1897.0              │
│ 75%        ┆ 2533.0       ┆ 2533.0              │
│ max        ┆ 457008.0     ┆ 50000.0             │
└────────────┴──────────────┴─────────────────────┘


## 5. Deduplikasi

Tentukan kolom kunci bisnis (misalnya ID unik) sebelum menghapus duplikat — jangan deduplikasi berdasarkan seluruh baris.

Kolom kunci pada dataset ini adalah `id`, penanda unik berita dari sumbernya.

In [6]:
KOLOM_KUNCI = 'id'

total_baris = len(df_outlier_handled)
baris_unik = df_outlier_handled[KOLOM_KUNCI].n_unique() if KOLOM_KUNCI in df_outlier_handled.columns else total_baris
print(f'Total baris: {total_baris:,}')
print(f'Baris unik berdasarkan {KOLOM_KUNCI}: {baris_unik:,}')
print(f'Duplikat: {total_baris - baris_unik:,}')

df_dedup = df_outlier_handled.unique(
    subset=[KOLOM_KUNCI] if KOLOM_KUNCI in df_outlier_handled.columns else None,
    keep='first',
    maintain_order=True,
)

print(f'Baris setelah deduplikasi: {len(df_dedup):,}')

Total baris: 640,000
Baris unik berdasarkan id: 640,000
Duplikat: 0
Baris setelah deduplikasi: 640,000


## 6. Validasi Data Dasar

Aturan yang dipakai, disesuaikan dengan isi dataset berita:

| Aturan | Batas | Alasan |
|---|---|---|
| Penulis terisi | tidak null | sudah diisi `Tanpa Penulis` |
| Tanggal dalam rentang | 2015 sampai 2026 | di luar itu nilai rusak, mis. tahun 2265 |
| Judul cukup panjang | minimal 10 karakter | judul 1 karakter tidak informatif |
| Isi cukup panjang | minimal 100 karakter | di bawah itu ekstraksi gagal |
| Panjang isi wajar | 100 sampai 50.000 karakter | batas bisnis, sudah dipotong |
| Kunci unik | `id` tidak berulang | satu baris satu berita |

In [7]:
connection.register('dataset_bersih', df_dedup)

validasi = connection.execute("""
    SELECT
        COUNT(*) AS total_baris,
        COUNT(DISTINCT id) AS id_unik,
        SUM(CASE WHEN news_author IS NULL THEN 1 ELSE 0 END) AS pelanggaran_penulis_null,
        SUM(CASE WHEN tanggal_terbit IS NULL THEN 1 ELSE 0 END) AS pelanggaran_tanggal_kosong,
        SUM(CASE WHEN tanggal_terbit IS NOT NULL
                  AND (YEAR(tanggal_terbit) < 2015 OR YEAR(tanggal_terbit) > 2026)
            THEN 1 ELSE 0 END) AS pelanggaran_tanggal_luar_rentang,
        SUM(CASE WHEN LENGTH(news_title) < 10 THEN 1 ELSE 0 END) AS pelanggaran_judul_pendek,
        SUM(CASE WHEN LENGTH(news_text) < 100 THEN 1 ELSE 0 END) AS pelanggaran_teks_pendek,
        SUM(CASE WHEN panjang_teks_bersih < 100
                  OR panjang_teks_bersih > 50000 THEN 1 ELSE 0 END) AS pelanggaran_panjang
    FROM dataset_bersih
""").pl()
validasi

total_baris,id_unik,pelanggaran_penulis_null,pelanggaran_tanggal_kosong,pelanggaran_tanggal_luar_rentang,pelanggaran_judul_pendek,pelanggaran_teks_pendek,pelanggaran_panjang
i64,i64,"decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]","decimal[38,0]"
640000,640000,0,16740,0,44,0,0


## 7. Simpan Data Bersih

Simpan hasil ke `data/processed/` agar dapat dipakai pada Milestone 3 tanpa mengulang proses cleaning.

In [8]:
import gc

# Semua langkah sebelumnya menahan salinan tabel di memori. Sebelum menulis,
# salinan antara dilepas dulu supaya proses simpan tidak kehabisan memori.
n_awal = len(df)
connection.unregister('dataset')
connection.unregister('dataset_bersih')
del df, df_missing_handled, df_teks, df_outlier_handled
gc.collect()

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# sink_parquet menulis bertahap, jadi tidak menahan seluruh tabel di memori
df_dedup.lazy().sink_parquet(OUTPUT_PATH)

print(f'Baris sebelum cleaning: {n_awal:,}')
print(f'Baris sesudah cleaning: {len(df_dedup):,}')
print(f'Data bersih tersimpan di: {OUTPUT_PATH}')
print(f'Ukuran berkas: {OUTPUT_PATH.stat().st_size / 1_000_000:,.1f} MB')

Baris sebelum cleaning: 640,000
Baris sesudah cleaning: 640,000
Data bersih tersimpan di: D:\Semester 7\Big Data\tugas1\data\processed\berita_politik_bersih.parquet
Ukuran berkas: 568.5 MB


## Catatan Milestone 2

**Dimensi kualitas yang paling bermasalah.** Kelengkapan (completeness) dan
validitas (validity). Dua kolom punya nilai kosong, dan salah satunya berisi
tanggal yang bentuknya sah tetapi isinya mustahil.

**Missing values.** `news_author` 31.931 baris (4,99%) diisi `Tanpa Penulis`.
`news_date` 16.624 baris (2,60%) tidak diisi nilai tebakan karena tanggal
karangan akan mencemari analisis temporal. Nilainya diubah ke datetime, lalu
116 baris yang tahunnya di luar 2015-2026 dijadikan null juga. Total yang
bertanda tidak valid 16.740 baris.

**Outlier.** Panjang isi berita punya sebaran lebar: terpendek 176 karakter,
terpanjang 457.008 karakter, median 1.897. Metode IQR menandai 18.816 baris di
atas batas 4.412 karakter, tetapi batas itu terlalu ketat untuk artikel berita
karena banyak artikel panjang yang tetap sah. Batas bisnis 100 sampai 50.000
karakter hanya menandai 4 baris, yaitu artikel yang isinya gabungan beberapa
halaman. Kolom turunan `panjang_teks_bersih` dipotong ke batas itu, isi berita
aslinya tidak diubah.

**Duplikat.** `id` dan `news_title` seluruhnya unik, jadi tidak ada baris yang
terhapus. Deduplikasi tetap dijalankan memakai `id` sebagai kunci, dan hasilnya
tetap 640.000 baris.

**Baris yang berubah.** Tidak ada baris yang dibuang. Jumlah baris tetap
640.000. Yang berubah: 31.931 penulis terisi, 16.740 tanggal jadi null
bertanda, dan 4 baris ditandai outlier panjang.

**Aturan validasi yang diterapkan.**

| Aturan | Pelanggaran |
|---|---|
| Penulis terisi | 0 |
| Tanggal dalam rentang 2015-2026 | 0 |
| Judul minimal 10 karakter | 44 |
| Isi minimal 100 karakter | 0 |
| Panjang isi 100-50.000 karakter | 0 |
| `id` unik | 0 |

Sisa 44 pelanggaran judul pendek dibiarkan, bukan dihapus, karena beritanya
tetap sah dan judulnya hanya perlu diperhatikan saat analisis teks. Kalau nanti
dibutuhkan, baris ini bisa disaring dengan `pl.col('panjang_judul') >= 10`.

**Kolom baru pada data bersih.** `tanggal_terbit` (datetime, null kalau tidak
valid), `tanggal_valid` (boolean), `panjang_teks`, `panjang_judul`,
`is_outlier_teks`, dan `panjang_teks_bersih`.